# Day 4: an evidence-driven ReAct coding agent

We will move from a fixed flow to an agent that gathers evidence, decides when it knows enough, proposes a repair, and then lets local Judge0 verify the result.

## 0. Start the services

Run these commands from the workshop root before opening this notebook:

```bash
source .venv/bin/activate
./workshop start
./workshop judge0-start
./workshop judge0-test
```

The model runs through the MLflow Gateway. Judge0 runs locally at `http://127.0.0.1:2358`.

In [ ]:
import json
import sqlite3
import sys
from pathlib import Path
from typing import Annotated, Literal, NotRequired, TypedDict

import mlflow
from langchain_core.messages import AIMessage, AnyMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from pydantic import BaseModel, Field

sys.path.append("../..")
from execution_tool import execute_code
from graph_image import show_graph

mlflow.set_tracking_uri("http://127.0.0.1:5001")
EXPERIMENT = "day-4-react-coding-agent"
mlflow.set_experiment(EXPERIMENT)

## 1. Why flows are not agents

![Technical comparison of a fixed LLM triage flow and an agent that chooses actions from observations](images/00-flow-vs-agent.svg)

A flow can contain an LLM and still be deterministic. In a triage flow, the author fixes the available branches, their order, and the stopping point. The model only supplies a value such as `logic` or `runtime`.

An agent owns part of the control policy: it chooses the next allowed action from the current state and observations. Code still supplies the boundaries—tools, sandbox, checkpoints, and a hard step limit.

## 2. Why ReAct became popular

![Technical flow showing the model decision, Judge0 tool action, structured observation, loop, and final answer](images/01-react-loop.svg)

ReAct uses one small loop: **decide → act → observe**. It became useful because tools stay ordinary functions, observations ground the next model call in external evidence, and the message history leaves an inspectable trace. The model can call a tool again or stop without requiring a custom graph for every possible path.

## 3. Start with problem, code, and testcases

When testcases are available, run them **one at a time**. After each result, decide whether the evidence is already strong enough or whether one more testcase is worth running. If there are no testcases, the agent can still inspect the problem and code, but its evidence is weaker.

`run_knapsack_test` accepts candidate code and one testcase number. The model cannot change the tests or Judge0 limits.

In [ ]:
problem = """
Given n items and a capacity, print the maximum value that fits.
Each item may be selected at most once.
""".strip()

buggy_code = """
n, capacity = map(int, input().split())
items = [tuple(map(int, input().split())) for _ in range(n)]
dp = [0] * (capacity + 1)
for weight, value in items:
    for current in range(weight, capacity + 1):
        dp[current] = max(dp[current], dp[current - weight] + value)
print(dp[capacity])
""".strip()

test_cases = [
    {"name": "two different items", "stdin": "2 5\n2 4\n3 5\n", "expected": "9\n"},
    {"name": "one item cannot repeat", "stdin": "1 4\n2 3\n", "expected": "3\n"},
    {"name": "best pair", "stdin": "3 6\n2 4\n3 5\n4 7\n", "expected": "11\n"},
]

In [ ]:
@tool
@mlflow.trace(name="react.run_knapsack_test", span_type="TOOL")
def run_knapsack_test(source_code: str, test_number: int) -> dict:
    """Run candidate code against one numbered visible testcase."""
    if not test_cases:
        return {
            "available": False,
            "reason": "No visible testcases were provided.",
        }

    if not 1 <= test_number <= len(test_cases):
        return {
            "available": False,
            "reason": f"Choose a testcase from 1 to {len(test_cases)}.",
        }

    case = test_cases[test_number - 1]
    outcome = execute_code(
        "python",
        source_code,
        case["stdin"],
        case["expected"],
    )
    return {
        "available": True,
        "number": test_number,
        "name": case["name"],
        "passed": outcome["passed"],
        "status": outcome["status"],
        "expected": case["expected"].strip(),
        "actual": outcome["stdout"],
        "compile_output": outcome["compile_output"],
    }

In [ ]:
print("Visible testcase numbers:", list(range(1, len(test_cases) + 1)))
print("No testcase has run yet. The agent will request them one at a time.")

The tool exposes the testcases as a queue, not a batch. The agent starts with test 1. If that passes, it should ask whether the passing case exercised the suspected bug before deciding to stop or run test 2.

## 4. Build the evidence loop

![Exact LangGraph topology with explicit RepairState, an agent node, ToolNode, structured proposal node, SQLiteSaver, and recursion limit](images/03-two-node-react-graph.svg)

`RepairState` makes every graph field visible. `bind_tools` gives the model an action it may choose, and `ToolNode` executes that request. A small router checks whether the latest model message contains a tool call.

When evidence is sufficient, the graph leaves the ReAct loop and calls a structured-output node. `RepairProposal` validates the diagnosis, confidence score, and fixed code before they enter state.

In [ ]:
llm = ChatOpenAI(
    base_url="http://127.0.0.1:5001/gateway/mlflow/v1",
    api_key="not-needed",
    model="workshop-vivek", # changed
    temperature=0,
)

class RepairState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    problem: str
    source_code: str
    test_cases: list[dict]
    diagnosis: NotRequired[str]
    confidence: NotRequired[float]
    fixed_code: NotRequired[str]


class RepairProposal(BaseModel):
    diagnosis: str = Field(description="Short explanation supported by observations")
    confidence: float = Field(ge=0, le=1)
    fixed_code: str = Field(description="Complete Python program without Markdown fences")


SYSTEM_PROMPT = """
You decide whether a Python repair needs more execution evidence. When visible
tests exist and none have run, call run_knapsack_test with test_number=1. Run
one untested testcase per call. If a test passes, run the next untested test.
Stop after a failing result strongly supports the code-level diagnosis. Do not
rerun the same testcase with unchanged code. Use at most two tool calls. If no
tests are available, reason from the problem and code.

When the evidence is sufficient, do not include the repair. Reply READY_TO_REPAIR.
Never claim a test passed unless a tool observation says it passed.
""".strip()

PROPOSAL_PROMPT = """
Return a RepairProposal for the buggy program. Base the diagnosis on the code and
observations. Confidence measures diagnostic confidence, not proof of correctness.
The fixed_code field must contain the complete program without Markdown fences.
""".strip()

model_with_tools = llm.bind_tools([run_knapsack_test])
proposal_model = llm.with_structured_output(RepairProposal)

In [ ]:
def problem_context(state: RepairState) -> HumanMessage:
    return HumanMessage(f"""
Problem:
{state['problem']}

Buggy Python program:
{state['source_code']}

Visible testcases:
{json.dumps(state['test_cases'], indent=2)}
""".strip())


@mlflow.trace(name="react.decide", span_type="AGENT")
def call_model(state: RepairState) -> dict:
    messages = [SystemMessage(SYSTEM_PROMPT), problem_context(state), *state["messages"]]
    return {"messages": [model_with_tools.invoke(messages)]}


def route_after_agent(state: RepairState) -> Literal["tools", "proposal"]:
    latest = state["messages"][-1]
    if isinstance(latest, AIMessage) and latest.tool_calls:
        return "tools"
    return "proposal"


@mlflow.trace(name="react.build_proposal", span_type="AGENT")
def build_proposal(state: RepairState) -> dict:
    messages = [SystemMessage(PROPOSAL_PROMPT), problem_context(state), *state["messages"]]
    proposal = proposal_model.invoke(messages)
    return {
        "diagnosis": proposal.diagnosis,
        "confidence": proposal.confidence,
        "fixed_code": proposal.fixed_code,
    }


builder = StateGraph(RepairState)
builder.add_node("agent", call_model)
builder.add_node("tools", ToolNode([run_knapsack_test]))
builder.add_node("proposal", build_proposal)
builder.add_edge(START, "agent")
builder.add_conditional_edges(
    "agent",
    route_after_agent,
    {"tools": "tools", "proposal": "proposal"},
)
builder.add_edge("tools", "agent")
builder.add_edge("proposal", END)

state_dir = Path("../../.workshop").resolve()
state_dir.mkdir(exist_ok=True)
checkpoint_db = state_dir / "day-4-react.sqlite"
checkpoint_connection = sqlite3.connect(checkpoint_db, check_same_thread=False)
checkpointer = SqliteSaver(checkpoint_connection)
react_agent = builder.compile(checkpointer=checkpointer)

show_graph(react_agent)

## 5. Let the agent decide when it has enough evidence

No testcase has run when the graph starts. The agent requests one testcase, observes one result, and decides again. When the evidence is sufficient, it returns `READY_TO_REPAIR`; the next node creates a validated `RepairProposal`.

In [ ]:
initial_state: RepairState = {
    "messages": [HumanMessage("Gather enough evidence to repair this program.")],
    "problem": problem,
    "source_code": buggy_code,
    "test_cases": test_cases,
}

THREAD_ID = "day-4-evidence-001"
config = {
    "configurable": {"thread_id": THREAD_ID},
    # Counts graph steps such as agent -> tool -> agent. It is a hard stop
    # for a runaway loop, not a testcase count or exact tool-call quota.
    # The independent verification below runs after this graph has ended.
    "recursion_limit": 8,
}

@mlflow.trace(name="day4.react_agent", span_type="AGENT")
def run_react_agent(state: RepairState) -> RepairState:
    mlflow.update_current_trace(
        session_id=THREAD_ID,
        tags={"day": "4", "thread_id": THREAD_ID},
    )
    return react_agent.invoke(state, config)

final_state = run_react_agent(initial_state)

In [ ]:
for message in final_state["messages"]:
    if isinstance(message, AIMessage) and message.tool_calls:
        names = [call["name"] for call in message.tool_calls]
        print("ACTION      ->", ", ".join(names))

    elif isinstance(message, ToolMessage):
        observation = json.loads(message.content)
        if not observation["available"]:
            print("OBSERVATION -> no visible testcases")
            continue
        print(
            "OBSERVATION ->",
            f"test {observation['number']}: {observation['status']} "
            f"(expected {observation['expected']}, actual {observation['actual']})",
        )

    elif isinstance(message, AIMessage):
        print("DECISION    ->", message.content)

print("\nSTRUCTURED OUTPUT")
print("diagnosis :", final_state["diagnosis"])
print("confidence:", final_state["confidence"])
print("fixed_code:\n", final_state["fixed_code"])

## 6. Verify the proposal outside the agent

![Technical pipeline from problem, code, and tests through evidence gathering, confidence-based exit, repair proposal, and deterministic Judge0 verification](images/04-evidence-repair-verify.svg)

Confidence controls whether the agent keeps investigating. It is not proof. Read the validated fields directly from `RepairState`, then run `fixed_code` against the visible suite outside the ReAct loop.

In [ ]:
proposed_code = final_state["fixed_code"]
confidence = final_state["confidence"]

@mlflow.trace(name="day4.verify_repair", span_type="CHAIN")
def verify_repair(source_code: str) -> dict:
    mlflow.update_current_trace(
        session_id=THREAD_ID,
        tags={"day": "4", "phase": "verification"},
    )
    results = []
    for test_number in range(1, len(test_cases) + 1):
        result = run_knapsack_test.invoke({
            "source_code": source_code,
            "test_number": test_number,
        })
        results.append(result)

    passed_count = sum(result["passed"] for result in results)
    return {
        "all_passed": bool(results) and passed_count == len(results),
        "passed_count": passed_count,
        "total": len(results),
        "tests": results,
    }

verification = verify_repair(proposed_code)

print("agent confidence:", confidence)
print("verified          :", verification["all_passed"])
print("visible tests     :", f"{verification['passed_count']}/{verification['total']}")

A good run ends with `verified: True`. If verification fails, that failure is a new observation for another repair loop—not something to hide behind a confidence score.

## 7. Inspect the checkpoint and trace

![Technical trace and checkpoint view showing ReAct decisions, structured RepairState fields, SQLite persistence, and recursion limit](images/05-react-checkpoints-traces.svg)

`recursion_limit=8` limits graph steps, not tool calls. SQLite stores the complete `RepairState`: messages, inputs, diagnosis, confidence, and fixed code. MLflow groups the agent trace and independent verification trace into one session because both use the same `THREAD_ID`.

In [ ]:
snapshot = react_agent.get_state(config)
print("saved messages  :", len(snapshot.values["messages"]))
print("saved confidence:", snapshot.values["confidence"])
print("saved diagnosis :", snapshot.values["diagnosis"])
print("next node       :", snapshot.next)
print("checkpoint      :", snapshot.config["configurable"]["checkpoint_id"][:8])

experiment = mlflow.get_experiment_by_name(EXPERIMENT)
traces = mlflow.search_traces(
    locations=[experiment.experiment_id],
    filter_string="tags.day = '4'",
    flush=True,
)
sessions = mlflow.search_sessions(
    locations=[experiment.experiment_id],
)
for session in sessions:
    print(f"session {session.id}: {len(session)} traces")

columns = [
    column
    for column in ["trace_id", "state", "execution_time_ms", "tags"]
    if column in traces.columns
]
traces[columns]

## Homework: generate stronger evidence

![Technical homework architecture where a generator creates valid inputs, a trusted oracle computes expected outputs, and Judge0 returns a counterexample](images/06-homework-counterexample.svg)

If confidence is low or the visible tests do not separate correct and wrong programs, add a `find_counterexample` tool or a second agentic flow:

1. Generate small valid inputs from a seed.
2. Compute expected output with a trusted brute-force oracle or property.
3. Run the candidate in Judge0 and compare outputs.
4. Return the smallest mismatch as a new observation.

An input generator alone is not enough: a testcase also needs a trusted expected result.

**Homework question:** Should test generation be another tool call inside this agent, or a separate bounded agent/flow? Defend the boundary you choose.

**Exit question:** What makes this an agent: access to Judge0, or deciding whether another observation is worth its cost?